# Multilingual Keyword Spotting — Hybrid TTS + MSWC Pipeline

**Final project — Intro to Deep Learning**

This notebook trains a small, edge-deployable wake-word model that recognises five **concept classes** (`silence`, `unknown`, `greeting`, `yes`, `no`) across six languages (English, German, Turkish, Arabic, French, Persian) using a **hybrid data pipeline**:

- **Primary source:** TTS audio generated with [`edge-tts`](https://github.com/rany2/edge-tts) (Microsoft Neural TTS, multiple voices per language).
- **Secondary source:** MSWC real samples where the inventory shows sufficient counts.
- **Held-out:** a slice of MSWC kept aside to measure the **synthetic → real domain gap**.

## Why this strategy?

MSWC is derived from Common Voice *read* text — even English `hello` has only ~300 samples, far too few to train across six languages. Lin et al. (ICASSP 2020) show that:

1. TTS alone can train usable KWS models (~57% with a DS-CNN, >90% with a frozen embedding + small head).
2. Augmenting TTS data (pitch/speed/reverb) does **not** help on top of voice diversity.

So our recipe is: **many voices, no on-waveform augmentation**, light SpecAugment in the spectrogram domain only as a regulariser.

## Model

DS-CNN (Zhang et al. 2017, *Hello Edge*) with a shared backbone, a 6-way **language head**, and a 5-way **keyword head conditioned on the language logits**. The conditioned design enables the *misrouting experiment*: forcing the wrong language identity into the keyword head and measuring how much accuracy degrades — quantifying how much the keyword head is *using* the language signal.

## Roadmap

1. Setup & config
2. TTS data generation (`edge-tts`)
3. Silence + unknown classes
4. MSWC supplement + domain-gap holdout
5. Log-mel feature extraction (49 × 40)
6. PyTorch `Dataset` and `DataLoader`
7. DS-CNN with conditioned heads
8. Multi-task training
9. Evaluation on TTS test split
10. Domain-gap evaluation on real MSWC
11. Misrouting experiment
12. Quantisation-Aware Training (QAT)
13. ONNX + TFLite export

## 1. Setup & config

In [ ]:
# Mount Google Drive for data persistence; pull latest code from GitHub.
# On local runs this cell is a no-op.
import sys
from pathlib import Path

REPO = "https://github.com/verilog-indeed/polyglot-kws.git"
REPO_DIR = Path("/content/polyglot-kws")

try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive/kws_data")
    if REPO_DIR.exists():
        !git -C {REPO_DIR} pull --quiet
        print("  updated repo from GitHub")
    else:
        !git clone --quiet {REPO} {REPO_DIR}
        print("  cloned repo from GitHub")
    SCRIPT_DIR = REPO_DIR
except ModuleNotFoundError:
    DRIVE_ROOT = None  # running locally
    SCRIPT_DIR = Path(".").resolve()

sys.path.insert(0, str(SCRIPT_DIR))


In [ ]:
# Nothing to install here for the training pipeline.
# Colab already ships with: torch, torchaudio, numpy, matplotlib, onnx (usually).
#
# Per-section installs happen inline, sandboxed, so a dep clash in one
# step can't break the rest of the notebook:
#   - Section 2  (TTS):   edge-tts librosa soundfile
#   - Section 13 (ONNX):  onnx onnxruntime onnxscript  (if missing in the runtime)
#   - Section 13 (TFLite): onnx2tf                  (heavy; pulls TF — see warning)
#
# If you already ran a broad install that pulled onnx2tf, restart the runtime
# (Runtime -> Restart session) before continuing — protobuf may have been
# downgraded by onnx/onnx2tf and that breaks tensorflow imports.

In [ ]:
import os, json, math, random, warnings
from datetime import datetime
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import torchaudio
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore', category=UserWarning)

SEED = 1234
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# ============================================================================
# DEBUG knob — flip to True to iterate fast locally.
#   - caps every (lang, kw) bucket at MAX_PER_BUCKET samples
#   - reduces EPOCHS in the training cell
# ============================================================================
DEBUG = False
MAX_PER_BUCKET    = 30   if DEBUG else 40    # cap per (lang, kw) bucket — balances unknown/silence
EPOCHS_DEFAULT    = 5    if DEBUG else 30
MAX_MSWC_PER_LANG = 50   if DEBUG else 400   # cap MSWC clips used per language
                                              # (independent of how many are on Drive)

# ----- paths -----
# ROOT auto-detects Colab vs local. Override with the KWS_ROOT env var if needed.
if Path('/content').exists():
    DEFAULT_ROOT = DRIVE_ROOT or Path('/content/kws_data')  # prefer Drive if mounted
else:
    DEFAULT_ROOT = Path.cwd() / 'kws_data'        # local (notebook cwd)
ROOT         = Path(os.environ.get('KWS_ROOT', str(DEFAULT_ROOT)))
TTS_DIR      = ROOT / 'tts'
MSWC_DIR     = ROOT / 'mswc'      # MSWC raw .opus/.wav files, if available
MANIFEST_DIR = ROOT / 'manifests'
CKPT_DIR     = ROOT / 'ckpts'
EXPORT_DIR   = ROOT / 'export'
for p in [TTS_DIR, MSWC_DIR, MANIFEST_DIR, CKPT_DIR, EXPORT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# Unique ID for this run — used to timestamp checkpoints and exports so
# re-runs don't overwrite previous results on Drive.
RUN_ID = datetime.now().strftime('%Y%m%d_%H%M')

# ----- audio config -----
SAMPLE_RATE  = 16000
WIN_MS, HOP_MS = 40, 20
N_MELS       = 40
N_FFT        = 1024
WIN_LENGTH   = int(SAMPLE_RATE * WIN_MS / 1000)   # 640
HOP_LENGTH   = int(SAMPLE_RATE * HOP_MS / 1000)   # 320
DURATION_S   = 1.0
NUM_SAMPLES  = int(SAMPLE_RATE * DURATION_S)      # 16000
NUM_FRAMES   = (NUM_SAMPLES - WIN_LENGTH) // HOP_LENGTH + 1  # 49

# ----- label spaces -----
LANGS    = ['en', 'de', 'tr', 'ar', 'fr', 'fa']
LANG2IDX = {l: i for i, l in enumerate(LANGS)}
KWS      = ['silence', 'unknown', 'activate', 'deactivate', 'play', 'stop']
KW2IDX   = {k: i for i, k in enumerate(KWS)}

SILENCE_LANG_IDX = -1
MSWC_KW_SENTINEL = -1

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'device           : {DEVICE}')
print(f'DEBUG            : {DEBUG}  (per-bucket cap={MAX_PER_BUCKET}, epochs={EPOCHS_DEFAULT})')
print(f'MAX_MSWC_PER_LANG: {MAX_MSWC_PER_LANG}')
print(f'ROOT             : {ROOT}')
print(f'RUN_ID           : {RUN_ID}')
print(f'feature shape    : ({N_MELS}, {NUM_FRAMES})')

## 2. TTS data generation (separate script)

To avoid dependency conflicts between `edge-tts` (and its `aiohttp` requirement) and the Torch / Colab runtime, **TTS generation lives in a stand-alone script**: [`generate_tts.py`](generate_tts.py). Run it once and the notebook simply reads the resulting WAVs from disk. The script also creates the silence class.

**Why TTS at all?** Lin et al. (ICASSP 2020) report ~57% accuracy with TTS-only data feeding a full DS-CNN (and >90% with TTS feeding a frozen embedding). Two practical takeaways:

1. **Voice diversity matters most.** Each voice contributes a slightly different formant pattern; pooling many voices gives better generalisation than augmenting fewer.
2. **Don't augment the waveform.** Pitch/speed/reverb augmentation gives no further gain on top of voice diversity.

We still apply mild **SpecAugment** later in training — it operates on the spectrogram and acts as a regulariser, not a domain-gap closer.

### Run the script

The next cell installs `edge-tts` + lightweight audio deps and runs the generation. It writes:

- `<ROOT>/tts/<lang>/<concept>/<wordslug>__<voice>.wav` — keyword and unknown classes
- `<ROOT>/tts/_silence/silence/silence_NNNN.wav` — silence class

The full corpus takes ~10–30 min over a typical network. Re-runs skip files that already exist.

In [ ]:
# Voice inventory loaded from shared config — single source of truth with generate_tts.py.
# sys.path is set by the Drive/GitHub cell above.
from kws_config import VOICES, KEYWORDS, VARIANTS, UNKNOWN_WORDS


In [ ]:
!pip install -q edge-tts librosa soundfile
!python {SCRIPT_DIR}/generate_tts.py --root {ROOT}


In [ ]:
# Pack all TTS/silence wavs into per-language .pt tensor bundles.
# ONE-TIME operation — after this runs, all subsequent sessions load a single
# .pt file per language instead of thousands of individual Drive files.
# Uses threads to parallelise the one-time Drive reads.
from concurrent.futures import ThreadPoolExecutor

def pack_tts_bundles(tts_dir: Path, overwrite: bool = False) -> None:
    for lang in LANGS:
        out = tts_dir / f'{lang}.pt'
        if out.exists() and not overwrite:
            n = torch.load(out, weights_only=False)['wavs'].shape[0]
            print(f'  [{lang}] already packed ({n} clips) — skipping')
            continue
        paths, kw_idxs, voices = [], [], []
        for concept in ('activate', 'deactivate', 'play', 'stop', 'unknown'):
            d = tts_dir / lang / concept
            if not d.exists():
                continue
            for p in sorted(d.glob('*.wav')):
                parts = p.stem.split('__')
                paths.append(p)
                kw_idxs.append(KW2IDX[concept])
                voices.append(parts[1] if len(parts) >= 2 else 'unknown_voice')
        if not paths:
            print(f'  [{lang}] no TTS wavs found — skipping')
            continue
        print(f'  [{lang}] packing {len(paths)} wavs ...', end=' ', flush=True)
        with ThreadPoolExecutor(max_workers=8) as ex:
            wavs = list(ex.map(load_wav, paths))
        torch.save({
            'wavs':    torch.stack(wavs),                          # (N, 1, 16000)
            'kw_idxs': torch.tensor(kw_idxs, dtype=torch.long),   # (N,)
            'voices':  voices,                                     # list[str]
        }, out)
        print('done')

    sil_out = tts_dir / '_silence.pt'
    if not sil_out.exists() or overwrite:
        sil_paths = sorted((tts_dir / '_silence' / 'silence').glob('*.wav'))
        if sil_paths:
            print(f'  silence: packing {len(sil_paths)} wavs ...', end=' ', flush=True)
            with ThreadPoolExecutor(max_workers=8) as ex:
                wavs = list(ex.map(load_wav, sil_paths))
            torch.save({'wavs': torch.stack(wavs)}, sil_out)
            print('done')

pack_tts_bundles(TTS_DIR)

## 3. About the silence and unknown classes

These are both produced by `generate_tts.py` above — covered here for reference:

- **Silence:** 1-second clips of mild Gaussian noise (`σ ≈ 0.005`). Pure zeros give the model nothing to learn from. Silence is language-agnostic, so we tag it with `lang_idx = -1` and **mask the language loss** for these samples via `CrossEntropyLoss(ignore_index=-1)`.
- **Unknown:** TTS-generated *non-keyword* commands (`record`, `cancel`, `launch`, `select`, …). The keyword head needs to learn to **reject** out-of-vocabulary speech, not just discriminate among the six concept classes.

## 4. MSWC supplement — real speech for language-head training

MSWC real samples serve a single purpose here: giving the **language identification head** exposure to genuine human speech, not just TTS voices. They carry **no keyword labels** — `kw_idx = -1` is assigned to every MSWC row and is masked out of the keyword loss via `CrossEntropyLoss(ignore_index=-1)`. Val and test sets are TTS-only (voice-disjoint).

### Getting the MSWC subset

```
pip install huggingface_hub soundfile torchaudio
python fetch_mswc.py --root <ROOT>
# default: 400 random clips per language, streamed directly from HF
```

No metadata file needed — the script streams each tar shard directly from HuggingFace and stops as soon as the per-language quota is met. Typically only 1–2 shards per language are touched.

A 50 % random holdout is reserved to measure **language accuracy on real speech** (the domain-gap experiment in section 10). If no MSWC data is present the pipeline still runs TTS-only; the domain-gap section is simply skipped.

In [ ]:
# One-shot helper: install huggingface_hub/soundfile if missing, then pull
# ~400 random MSWC clips per language into <ROOT>/mswc/<lang>/clip_NNNNN.wav.
# No metadata file needed — word labels are irrelevant (kw_idx=-1 for all MSWC rows).
# Comment out if you've already populated MSWC, or just don't want any real data.
try:
    import huggingface_hub, soundfile  # noqa: F401
except ImportError:
    !pip install -q huggingface_hub soundfile

!python {SCRIPT_DIR}/fetch_mswc.py --root {ROOT} --per-lang 400

In [ ]:
# MSWC provides language supervision only — no keyword alignment needed.
# kw_idx=-1 is assigned to all MSWC rows (masked from keyword loss in training).

MIN_MSWC_FOR_TRAIN = 5 if DEBUG else 10
MSWC_HOLDOUT_FRAC  = 0.5

def discover_mswc() -> dict:
    """Return {lang: FloatTensor(N, 1, 16000)} from per-language data.pt bundles.
    Falls back to packing existing individual wavs if data.pt is absent (migration)."""
    from concurrent.futures import ThreadPoolExecutor
    found = {}
    for lang in LANGS:
        data_pt = MSWC_DIR / lang / 'data.pt'
        wav_dir = MSWC_DIR / lang

        if data_pt.exists():
            found[lang] = torch.load(data_pt, weights_only=False)['wavs']
            continue

        # Migration: old-format individual wavs → pack once into data.pt
        existing = sorted(wav_dir.glob('clip_*.wav')) if wav_dir.exists() else []
        if existing:
            print(f'  [{lang}] migrating {len(existing)} wavs → data.pt ...', end=' ', flush=True)
            with ThreadPoolExecutor(max_workers=8) as ex:
                tensors = list(ex.map(load_wav, existing))
            bundle = torch.stack(tensors)
            data_pt.parent.mkdir(parents=True, exist_ok=True)
            torch.save({'wavs': bundle}, data_pt)
            print('done')
            found[lang] = bundle

    return found

mswc_pool = discover_mswc()
print('MSWC inventory (lang → clip count):')
total = 0
for lang in LANGS:
    n = mswc_pool.get(lang, torch.zeros(0)).shape[0]
    if n:
        print(f'  {lang}: {n}')
        total += n
print(f'total MSWC clips loaded: {total}')
if total == 0:
    print('  (no MSWC data — run fetch_mswc.py, or skip; pipeline runs TTS-only)')

## 5. Log-mel feature extraction (49 × 40)

Each 1-second clip becomes a **49 × 40** image:

- 16 kHz audio → 16 000 samples per clip.
- 40 ms window (640 samples), 20 ms hop (320 samples) → `(16 000 − 640) / 320 + 1 = 49` frames.
- 40 mel filterbanks spanning 0 Hz → Nyquist.
- Log-amplitude (dB, `top_db=80`), then normalised to ~[0, 1] via `(mel_db + 80) / 80`.

These dimensions match TensorFlow Lite Micro's *microfrontend* convention, which is what the ESP32 deployment path expects.

In [ ]:
mel_transform = torchaudio.transforms.MelSpectrogram(
    sample_rate=SAMPLE_RATE, n_fft=N_FFT, win_length=WIN_LENGTH,
    hop_length=HOP_LENGTH, n_mels=N_MELS, center=False,
).to(DEVICE)
amp2db = torchaudio.transforms.AmplitudeToDB(stype='power', top_db=80).to(DEVICE)

def load_wav(path) -> torch.Tensor:
    """Load any audio at SAMPLE_RATE, mono, padded/cropped to NUM_SAMPLES."""
    wav, sr = torchaudio.load(str(path))
    wav = wav.mean(dim=0, keepdim=True)
    if sr != SAMPLE_RATE:
        wav = torchaudio.functional.resample(wav, sr, SAMPLE_RATE)
    n = wav.shape[-1]
    if n >= NUM_SAMPLES:
        wav = wav[..., :NUM_SAMPLES]
    else:
        wav = F.pad(wav, (0, NUM_SAMPLES - n))
    return wav  # (1, NUM_SAMPLES)

def waveform_to_logmel(wav: torch.Tensor) -> torch.Tensor:
    """(..., NUM_SAMPLES) → (..., NUM_FRAMES, N_MELS) normalised to ~[0, 1].
    Works on single samples (1, S) or batches (B, 1, S); input must be on DEVICE."""
    mel = mel_transform(wav)            # (..., N_MELS, NUM_FRAMES)
    mel = amp2db(mel)
    mel = (mel + 80.0) / 80.0
    return mel.transpose(-1, -2)        # (..., NUM_FRAMES, N_MELS)

In [ ]:
# Sanity-check: pick one TTS sample and plot its log-mel.
sample_files = list((TTS_DIR / 'en' / 'activate').glob('*.wav'))
if sample_files:
    w = load_wav(sample_files[0]).to(DEVICE)
    m = waveform_to_logmel(w).squeeze(0).cpu().numpy()
    fig, ax = plt.subplots(figsize=(5, 2.2))
    ax.imshow(m.T, origin='lower', aspect='auto')
    ax.set_xlabel('frame (20 ms)'); ax.set_ylabel('mel bin')
    ax.set_title(f'log-mel — {sample_files[0].name}')
    plt.tight_layout(); plt.show()
else:
    print('No TTS samples yet — run cell 7 first.')

## 6. Dataset and DataLoader

We build a **manifest**: a list of `(path, lang_idx, kw_idx, voice_id)` tuples split into:

- `train` — TTS train voices + (optionally) MSWC train supplement
- `val` — TTS validation voices (held-out)
- `tts_test` — TTS test voices (held-out, never seen during training)
- `mswc_test` — MSWC held-out real samples → domain-gap test

Splits are done **voice-disjoint** for TTS so the model can't memorise voice timbres.

In [ ]:
def _cap_per_bucket(rows, cap):
    """Optionally cap rows to `cap` per (lang_idx, kw_idx) bucket. Deterministic."""
    if cap is None:
        return rows
    rng = random.Random(SEED + 3)
    buckets = defaultdict(list)
    for r in rows:
        buckets[(r[1], r[2])].append(r)
    capped = []
    for k, v in buckets.items():
        rng.shuffle(v)
        capped.extend(v[:cap])
    return capped

# ── Build flat WAV_CACHE ────────────────────────────────────────────────────
# All audio is pre-loaded here into one contiguous tensor. KWSDataset.__getitem__
# becomes a pure index lookup — zero disk I/O during training or evaluation.
# Workers inherit WAV_CACHE via fork (copy-on-write on Linux); reads never copy.

_wav_chunks: list = []
_wav_offset: int  = 0

def _register(wavs: torch.Tensor) -> int:
    """Append a block of wavs to the cache; return its starting index."""
    global _wav_offset
    start = _wav_offset
    _wav_chunks.append(wavs)
    _wav_offset += wavs.shape[0]
    return start

def collect_tts():
    rows = []
    for lang in LANGS:
        bp = TTS_DIR / f'{lang}.pt'
        if not bp.exists():
            continue
        b     = torch.load(bp, weights_only=False)
        start = _register(b['wavs'])
        for i in range(b['wavs'].shape[0]):
            rows.append((start + i, LANG2IDX[lang], b['kw_idxs'][i].item(), b['voices'][i]))

    sp = TTS_DIR / '_silence.pt'
    if sp.exists():
        b = torch.load(sp, weights_only=False)
        n = b['wavs'].shape[0]
        # Silence has one "voice" so voice-disjoint split can't apply.
        # Assign synthetic split tags by shuffled index: 80% train, 10% val, 10% test.
        rng_sil = random.Random(SEED + 99)
        idx = list(range(n))
        rng_sil.shuffle(idx)
        n_test = max(1, int(n * 0.10))
        n_val  = max(1, int(n * 0.10))
        tags = ['silence_train'] * n
        for i in idx[:n_test]:
            tags[i] = 'silence_test'
        for i in idx[n_test:n_test + n_val]:
            tags[i] = 'silence_val'
        start = _register(b['wavs'])
        for i in range(n):
            rows.append((start + i, SILENCE_LANG_IDX, KW2IDX['silence'], tags[i]))
    return rows

def split_tts_by_voice(rows, val_per_lang=1, test_per_lang=1):
    rng = random.Random(SEED + 13)
    val_set, test_set = set(), set()
    for lang in LANGS:
        voices = sorted(VOICES[lang])
        rng.shuffle(voices)
        n_test = min(test_per_lang, max(0, len(voices) - 1))
        n_val  = min(val_per_lang,  max(0, len(voices) - n_test - 1))
        test_set.update(voices[:n_test])
        val_set.update(voices[n_test:n_test + n_val])
    # Silence synthetic tags route into the correct split
    val_set.add('silence_val')
    test_set.add('silence_test')
    train, val, test = [], [], []
    for row in rows:
        v = row[3]
        if v in test_set:   test.append(row)
        elif v in val_set:  val.append(row)
        else:               train.append(row)
    return train, val, test

def collect_mswc(pool, holdout_frac=MSWC_HOLDOUT_FRAC, min_train=MIN_MSWC_FOR_TRAIN):
    """MSWC provides language supervision only (kw_idx=-1, masked from keyword loss).
    MAX_MSWC_PER_LANG caps usage; only the needed subset is registered in WAV_CACHE."""
    rng = random.Random(SEED + 7)
    train_rows, held_rows = [], []
    for lang, wavs in pool.items():
        indices = list(range(wavs.shape[0]))
        rng.shuffle(indices)
        if MAX_MSWC_PER_LANG is not None:
            indices = indices[:MAX_MSWC_PER_LANG]
        subset = wavs[indices]          # only register the clips we actually use
        start  = _register(subset)
        cut    = int(len(indices) * holdout_frac)
        for i in range(cut):
            held_rows.append((start + i, LANG2IDX[lang], MSWC_KW_SENTINEL, f'mswc_{lang}'))
        if len(indices) - cut >= min_train:
            for i in range(cut, len(indices)):
                train_rows.append((start + i, LANG2IDX[lang], MSWC_KW_SENTINEL, f'mswc_{lang}'))
    return train_rows, held_rows

tts_rows = collect_tts()
tts_train, tts_val, tts_test = split_tts_by_voice(tts_rows)
mswc_train, mswc_held = collect_mswc(mswc_pool)

WAV_CACHE = torch.cat(_wav_chunks, dim=0) if _wav_chunks else torch.zeros(0, 1, NUM_SAMPLES)
print(f'WAV_CACHE: {tuple(WAV_CACHE.shape)}  ({WAV_CACHE.nbytes / 1e9:.2f} GB)')

manifest = {
    # Cap TTS rows only — MSWC rows (kw_idx=-1) must not be capped, they are
    # the language head's primary training signal and share a single bucket key.
    'train':     _cap_per_bucket(tts_train, MAX_PER_BUCKET) + mswc_train,
    'val':       _cap_per_bucket(tts_val,   MAX_PER_BUCKET),
    'tts_test':  _cap_per_bucket(tts_test,  MAX_PER_BUCKET),
    'mswc_held': mswc_held,
}
for name, rows in manifest.items():
    cnt = Counter((r[1], r[2]) for r in rows)
    print(f'{name:10s}  total={len(rows):4d}   unique (lang,kw)={len(cnt)}')

In [ ]:
class SpecAugment:
    """Light SpecAugment in the (frame, mel) domain. Applied on GPU in the training loop."""
    def __init__(self, freq_mask=8, time_mask=12, n_masks=2):
        self.fm, self.tm, self.n = freq_mask, time_mask, n_masks
    def __call__(self, m):  # m: (1, T, F)
        T, Fb = m.shape[-2], m.shape[-1]
        for _ in range(self.n):
            f  = random.randint(0, self.fm)
            f0 = random.randint(0, max(0, Fb - f))
            m[..., :, f0:f0 + f] = 0
            t  = random.randint(0, self.tm)
            t0 = random.randint(0, max(0, T - t))
            m[..., t0:t0 + t, :] = 0
        return m

_spec_aug = SpecAugment()

class KWSDataset(Dataset):
    """Indexes into the pre-built WAV_CACHE — no file I/O at any point."""
    def __init__(self, rows, augment=False):
        self.rows    = rows
        self.augment = augment
    def __len__(self): return len(self.rows)
    def __getitem__(self, i):
        wav_idx, lang_idx, kw_idx, _ = self.rows[i]
        return WAV_CACHE[wav_idx], torch.tensor(kw_idx), torch.tensor(lang_idx)

def make_loaders(batch_size=128, num_workers=4):
    out = {}
    for k, rows in manifest.items():
        if not rows: continue
        out[k] = DataLoader(
            KWSDataset(rows, augment=(k == 'train')),
            batch_size=batch_size, shuffle=(k == 'train'),
            num_workers=num_workers, drop_last=(k == 'train'),
            pin_memory=(DEVICE == 'cuda'),
            persistent_workers=(num_workers > 0),
            prefetch_factor=2,
        )
    return out

loaders = make_loaders()
print({k: len(v.dataset) for k, v in loaders.items()})

## 7. DS-CNN with conditioned heads

A **depthwise-separable conv** splits a standard 3×3 conv into a 3×3 *depthwise* (one filter per channel) followed by a 1×1 *pointwise* (mixes channels). For our channel sizes this drops parameter count by ~8× with little accuracy loss — exactly what we want for an edge model.

Two heads share the backbone:

- **Language head:** `Linear(128, 6)` — predicts which of the six languages the clip is.
- **Keyword head:** receives backbone features concatenated with the *detached* language logits → `Linear(128 + 6, 64) → ReLU → Linear(64, 5)`.

The `.detach()` prevents the keyword loss from corrupting the language head. The conditioning gives the keyword head a prior over language identity — essential for disambiguating *salam / selam / سلام* across Turkish, Arabic and Persian, where the acoustic signal alone is genuinely ambiguous.

In [ ]:
class DSBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.dw  = nn.Conv2d(in_ch, in_ch, 3, stride=stride, padding=1, groups=in_ch, bias=False)
        self.bn1 = nn.BatchNorm2d(in_ch)
        self.pw  = nn.Conv2d(in_ch, out_ch, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)
    def forward(self, x):
        x = F.relu(self.bn1(self.dw(x)))
        x = F.relu(self.bn2(self.pw(x)))
        return x

class MultilingualKWS(nn.Module):
    def __init__(self, n_langs=len(LANGS), n_kw=len(KWS)):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Conv2d(1, 64, 3, padding=1, bias=False),
            nn.BatchNorm2d(64), nn.ReLU(),
            DSBlock(64, 64), DSBlock(64, 64),
            nn.MaxPool2d(2),
            DSBlock(64, 128, stride=2), DSBlock(128, 128),
            nn.AdaptiveAvgPool2d(1),
        )
        self.lang_head = nn.Linear(128, n_langs)
        self.kw_head = nn.Sequential(
            nn.Linear(128 + n_langs, 64), nn.ReLU(),
            nn.Linear(64, n_kw),
        )
    def forward(self, x, forced_lang_logits=None):
        feat = self.backbone(x).flatten(1)           # (B, 128)
        lang_logits = self.lang_head(feat)           # (B, n_langs)
        cond = forced_lang_logits if forced_lang_logits is not None else lang_logits.detach()
        kw_logits = self.kw_head(torch.cat([feat, cond], dim=1))
        return kw_logits, lang_logits

_m = MultilingualKWS()
n_params = sum(p.numel() for p in _m.parameters() if p.requires_grad)
print(f'trainable params: {n_params:,}   ({n_params/1e3:.1f}K)')
del _m

## 8. Multi-task training

Loss:

$$\mathcal{L}_{\text{total}} = \mathcal{L}_{\text{keyword}} + \lambda \cdot \mathcal{L}_{\text{language}}, \quad \lambda = 0.5$$

`L_language` uses `ignore_index=-1` to skip silence samples (their language is undefined). We track keyword accuracy on val and save the best checkpoint.

In [ ]:
LAMBDA_LANG = 1.0   # increased from 0.5 — language head was undertrained
EPOCHS      = EPOCHS_DEFAULT   # set by DEBUG flag in section 1
LR          = 1e-3
WD          = 1e-4

def epoch_pass(model, loader, opt=None, sched=None):
    train = opt is not None
    model.train(train)
    ce_kw  = nn.CrossEntropyLoss(ignore_index=-1)
    ce_lng = nn.CrossEntropyLoss(ignore_index=-1)
    total = n_kw_ok = total_kw = n_lng_ok = n_lng_total = 0
    loss_sum = 0.0
    augment = train and loader.dataset.augment
    for wav, kw, lang in loader:
        wav  = wav.to(DEVICE, non_blocking=True)
        kw   = kw.to(DEVICE, non_blocking=True)
        lang = lang.to(DEVICE, non_blocking=True)
        mel = waveform_to_logmel(wav)           # (B, 1, NUM_FRAMES, N_MELS)
        if augment:
            mel = mel.clone()
            for b in range(mel.size(0)):
                mel[b] = _spec_aug(mel[b])
        mel = mel.transpose(-1, -2)             # (B, 1, N_MELS, NUM_FRAMES) for Conv2d
        with torch.set_grad_enabled(train):
            kw_l, lng_l = model(mel)
            loss = ce_kw(kw_l, kw) + LAMBDA_LANG * ce_lng(lng_l, lang)
            loss = torch.nan_to_num(loss, nan=0.0)  # guard all-masked batches
        if train:
            opt.zero_grad(); loss.backward(); opt.step()
            if sched is not None: sched.step()
        total       += kw.size(0)
        kw_mask      = (kw != -1)
        total_kw    += kw_mask.sum().item()
        n_kw_ok     += ((kw_l.argmax(1) == kw) & kw_mask).sum().item()
        mask         = (lang != -1)
        n_lng_total += mask.sum().item()
        n_lng_ok    += ((lng_l.argmax(1) == lang) & mask).sum().item()
        loss_sum    += loss.item() * kw.size(0)
    return {
        'loss':     loss_sum / max(1, total),
        'kw_acc':   n_kw_ok / max(1, total_kw),
        'lang_acc': n_lng_ok / max(1, n_lng_total),
    }

def train_model():
    m   = MultilingualKWS().to(DEVICE)
    if hasattr(torch, 'compile'):
        m = torch.compile(m, mode='reduce-overhead')
    opt = torch.optim.AdamW(m.parameters(), lr=LR, weight_decay=WD)
    steps = max(1, EPOCHS * len(loaders['train']))
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=steps, pct_start=0.15)
    best_val, best_state = -1, None
    best_path = CKPT_DIR / f'best_{RUN_ID}.pt'
    last_path = CKPT_DIR / 'last.pt'       # overwritten every epoch — Drive flush safety net
    for ep in range(1, EPOCHS + 1):
        tr = epoch_pass(m, loaders['train'], opt, sched)
        va = epoch_pass(m, loaders['val'])
        print(f'ep {ep:02d}  train loss {tr["loss"]:.3f}  kw {tr["kw_acc"]:.3f}  lang {tr["lang_acc"]:.3f}  | '
              f'val kw {va["kw_acc"]:.3f}  lang {va["lang_acc"]:.3f}')
        current_state = {k.replace('_orig_mod.', ''): v.detach().cpu().clone()
                         for k, v in m.state_dict().items()}
        torch.save(current_state, last_path)   # always save latest
        if va['kw_acc'] > best_val:
            best_val   = va['kw_acc']
            best_state = current_state
            torch.save(best_state, best_path)
            print(f'       ↑ new best — saved to {best_path.name}')
    m.load_state_dict({f'_orig_mod.{k}': v for k, v in best_state.items()}, strict=False)
    print(f'\nbest val kw_acc = {best_val:.3f}  (epochs={EPOCHS})')
    return m

model = train_model()

## 9. Evaluation on the TTS test split

We measure keyword and language accuracy on the **held-out voices** — voices the model never saw during training. This is the clean in-domain test.

In [ ]:
@torch.no_grad()
def evaluate(model, loader, name):
    model.eval()
    kw_p, kw_t, lng_p, lng_t = [], [], [], []
    for wav, kw, lang in loader:
        wav = wav.to(DEVICE, non_blocking=True)
        mel = waveform_to_logmel(wav).transpose(-1, -2)  # (B, 1, F, T)
        kw_l, lng_l = model(mel)
        kw_mask = (kw != -1)
        kw_p += kw_l.argmax(1)[kw_mask].cpu().tolist()
        kw_t += kw[kw_mask].cpu().tolist()
        for p, t in zip(lng_l.argmax(1).cpu().tolist(), lang.tolist()):
            if t != -1:
                lng_p.append(p); lng_t.append(t)
    kw_acc   = float(np.mean(np.array(kw_p) == np.array(kw_t))) if kw_t else float('nan')
    lang_acc = float(np.mean(np.array(lng_p) == np.array(lng_t))) if lng_t else float('nan')
    print(f'[{name}] kw_acc={kw_acc:.3f}  lang_acc={lang_acc:.3f}  (n_kw={len(kw_t)}, n_lang={len(lng_t)})')
    return {'kw_pred': kw_p, 'kw_true': kw_t,
            'lang_pred': lng_p, 'lang_true': lng_t,
            'kw_acc': kw_acc, 'lang_acc': lang_acc}

tts_results = evaluate(model, loaders['tts_test'], 'TTS test')

In [ ]:
def plot_confusion(true, pred, labels, title):
    n  = len(labels)
    cm = np.zeros((n, n), dtype=int)
    for t, p in zip(true, pred):
        cm[t, p] += 1
    cmn = cm / cm.sum(axis=1, keepdims=True).clip(min=1)
    fig, ax = plt.subplots(figsize=(4 + 0.2 * n, 3.5 + 0.2 * n))
    im = ax.imshow(cmn, vmin=0, vmax=1, cmap='Blues')
    ax.set_xticks(range(n)); ax.set_yticks(range(n))
    ax.set_xticklabels(labels, rotation=30, ha='right'); ax.set_yticklabels(labels)
    for i in range(n):
        for j in range(n):
            ax.text(j, i, f'{cm[i, j]}', ha='center', va='center',
                    color='white' if cmn[i, j] > 0.5 else 'black', fontsize=8)
    ax.set_xlabel('predicted'); ax.set_ylabel('true'); ax.set_title(title)
    plt.colorbar(im, ax=ax); plt.tight_layout(); plt.show()

plot_confusion(tts_results['kw_true'],   tts_results['kw_pred'],   KWS,   'Keyword — TTS test')
plot_confusion(tts_results['lang_true'], tts_results['lang_pred'], LANGS, 'Language — TTS test')

## 10. Domain-gap evaluation — language accuracy on real speech

MSWC samples carry only language labels; the keyword head sees no MSWC supervision. This section evaluates whether the **language head** generalises from TTS voices to real human speech. The gap between TTS and real-speech language accuracy quantifies how well the language embeddings transfer across the synthetic/real domain boundary.

A large gap suggests the language head has overfit to TTS prosody or voice characteristics; a small gap means the learned language representations are acoustic-phonetically grounded.

In [ ]:
if 'mswc_held' in loaders and len(loaders['mswc_held'].dataset) > 0:
    model.eval()
    lng_p, lng_t = [], []
    with torch.no_grad():
        for wav, kw, lang in loaders['mswc_held']:
            wav = wav.to(DEVICE, non_blocking=True)
            mel = waveform_to_logmel(wav).transpose(-1, -2)
            _, lng_l = model(mel)
            for p, t in zip(lng_l.argmax(1).cpu().tolist(), lang.tolist()):
                if t != -1:
                    lng_p.append(p); lng_t.append(t)

    real_lang_acc = float(np.mean(np.array(lng_p) == np.array(lng_t))) if lng_t else float('nan')
    tts_lang_acc  = tts_results['lang_acc']
    print(f'Language accuracy — TTS test : {tts_lang_acc:.3f}')
    print(f'Language accuracy — real MSWC: {real_lang_acc:.3f}')
    print(f'=> domain gap (TTS − real)   : {tts_lang_acc - real_lang_acc:+.3f}')

    if lng_t:
        plot_confusion(lng_t, lng_p, LANGS, 'Language head — real MSWC speech')
else:
    print('No MSWC held-out samples available; skipping language domain-gap evaluation.')

## 11. Misrouting experiment — the core contribution

We force the language head to output a specific (possibly wrong) language identity and observe what happens to keyword accuracy.

**Procedure.** For every `(true_lang, forced_lang)` pair, build a one-hot logit vector with `+10` on the forced language and `0` elsewhere, feed it as `forced_lang_logits`, and measure keyword accuracy on the TTS test set.

**Expectation.** With phonetically distinct command words across languages (`activate` / `aktivieren` / `etkinleştir` / `فعّل` / `activer` / `فعال کن`), the keyword head may already be robust to language misrouting — the model learns acoustic representations rather than language-conditioned shortcuts. The misrouting matrix directly quantifies this:

- High off-diagonal accuracy → keyword representations are language-agnostic; the conditioning is redundant but harmless.
- Low off-diagonal accuracy → the keyword head is actively using the language signal; the conditioned design is earning its keep (most likely for acoustically similar pairs across related language groups).

Either outcome is a publishable finding.

In [ ]:
@torch.no_grad()
def misroute(model, loader, n_langs=len(LANGS)):
    model.eval()
    acc = np.full((n_langs, n_langs), np.nan)
    for forced in range(n_langs):
        per_true = defaultdict(lambda: [0, 0])  # true_lang → [correct, total]
        for wav, kw, lang in loader:
            wav = wav.to(DEVICE, non_blocking=True)
            kw  = kw.to(DEVICE, non_blocking=True)
            mel = waveform_to_logmel(wav).transpose(-1, -2)
            fl  = torch.zeros(mel.size(0), n_langs, device=DEVICE)
            fl[:, forced] = 10.0
            kw_l, _ = model(mel, forced_lang_logits=fl)
            pred = kw_l.argmax(1).cpu()
            for kp, kt, lt in zip(pred.tolist(), kw.cpu().tolist(), lang.tolist()):
                if lt == -1: continue
                per_true[lt][1] += 1
                per_true[lt][0] += int(kp == kt)
        for t in range(n_langs):
            c, n = per_true[t]
            if n: acc[t, forced] = c / n
    return acc

misroute_acc = misroute(model, loaders['tts_test'])
print('rows = true language, cols = forced language')
print(np.round(misroute_acc, 3))

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(misroute_acc, cmap='RdYlGn', vmin=0, vmax=1)
ax.set_xticks(range(len(LANGS))); ax.set_yticks(range(len(LANGS)))
ax.set_xticklabels(LANGS); ax.set_yticklabels(LANGS)
ax.set_xlabel('forced language'); ax.set_ylabel('true language')
ax.set_title('Keyword accuracy under language misrouting')
for i in range(len(LANGS)):
    for j in range(len(LANGS)):
        v = misroute_acc[i, j]
        ax.text(j, i, f'{v:.2f}' if not np.isnan(v) else '–',
                ha='center', va='center', fontsize=8)
plt.colorbar(im, ax=ax); plt.tight_layout(); plt.show()

## 12. Quantisation-Aware Training (QAT)

To run on an ESP32-S3 (or any 8-bit MCU), the model must be **integer-quantised**. QAT inserts *fake-quantisation* nodes during training so the model learns weights and activations that survive the int8 rounding.

We use PyTorch's `torch.ao.quantization` with the QNNPACK backend (ARM-targeted). The flow:

1. Wrap the float model in quant/dequant stubs.
2. Set the QAT qconfig and call `prepare_qat` — observers + fake-quant inserted.
3. Fine-tune at a low LR for a few epochs.
4. `convert` to produce an int8 model.

*Note:* a tutorial-level setup skips manual Conv–BN fusion; for a tighter int8 model add `torch.ao.quantization.fuse_modules` calls before `prepare_qat`.

In [ ]:
import torch.ao.quantization as tq

class QATWrapper(nn.Module):
    def __init__(self, inner):
        super().__init__()
        self.quant = tq.QuantStub()
        self.dq_kw   = tq.DeQuantStub()
        self.dq_lang = tq.DeQuantStub()
        self.inner = inner
    def forward(self, x):
        x = self.quant(x)
        kw, lang = self.inner(x)
        return self.dq_kw(kw), self.dq_lang(lang)

torch.backends.quantized.engine = 'qnnpack'
qat_model = QATWrapper(MultilingualKWS())
# torch.compile adds '_orig_mod.' prefix to all state dict keys — strip it before loading
qat_model.inner.load_state_dict(
    {k.replace('_orig_mod.', ''): v.cpu() for k, v in model.state_dict().items()}
)
qat_model.qconfig = tq.get_default_qat_qconfig('qnnpack')
qat_model = tq.prepare_qat(qat_model.train(), inplace=False)
qat_model = qat_model.to(DEVICE)

QAT_EPOCHS = 3
opt = torch.optim.AdamW(qat_model.parameters(), lr=1e-4)
ce_kw  = nn.CrossEntropyLoss(ignore_index=-1)
ce_lng = nn.CrossEntropyLoss(ignore_index=-1)
for ep in range(QAT_EPOCHS):
    qat_model.train()
    for wav, kw, lang in loaders['train']:
        wav  = wav.to(DEVICE, non_blocking=True)
        kw   = kw.to(DEVICE, non_blocking=True)
        lang = lang.to(DEVICE, non_blocking=True)
        mel  = waveform_to_logmel(wav).transpose(-1, -2)
        opt.zero_grad()
        kw_l, lng_l = qat_model(mel)
        loss = ce_kw(kw_l, kw) + LAMBDA_LANG * ce_lng(lng_l, lang)
        loss = torch.nan_to_num(loss, nan=0.0)
        loss.backward(); opt.step()
    print(f'QAT epoch {ep+1}/{QAT_EPOCHS} done')

qat_model.eval()
int8_model = tq.convert(qat_model.cpu(), inplace=False)  # qnnpack requires CPU for convert
int8_path  = CKPT_DIR / f'int8_{RUN_ID}.pt'
torch.save(int8_model.state_dict(), int8_path)
print('int8 model saved →', int8_path.name)

## 13. ONNX + TFLite export

For ESP32 deployment we need a **TFLite int8** model. The cleanest path from PyTorch is:

1. Export the float model to **ONNX** (`torch.onnx.export`).
2. Use **onnx2tf** to convert ONNX → TF saved_model → TFLite.
3. Apply post-training int8 quantisation in the TFLite converter with a representative dataset at deployment time.

### Dep note

`onnx` + `onnxruntime` are usually already in Colab — the cell below installs only if missing. `onnx2tf` is heavy (pulls `tensorflow`) and is the most likely thing to perturb the runtime's `protobuf` / `numpy` pins. If pip prints conflict warnings, that's almost always harmless for our needs — what matters is that the `import` calls in the cells below succeed. If they don't, restart the runtime and re-run only the export cells (you can reload the trained checkpoint from `CKPT_DIR/'best.pt'`).

We export the inference path (no `forced_lang_logits` argument) so the ONNX graph has a single input.

In [ ]:
try:
    import onnx, onnxruntime  # noqa: F401  — onnxscript excluded: incompatible with torch.compile
except ImportError:
    !pip install -q --upgrade onnx onnxruntime

import onnxruntime as ort

class InferenceModel(nn.Module):
    def __init__(self, inner):
        super().__init__()
        self.inner = inner
    def forward(self, x):
        return self.inner(x)  # (kw_logits, lang_logits)

infer = InferenceModel(MultilingualKWS()).eval()
# Load from the saved checkpoint — clean keys, no torch.compile/_orig_mod. prefix
state = torch.load(CKPT_DIR / f'best_{RUN_ID}.pt', weights_only=True)
infer.inner.load_state_dict(state)

dummy     = torch.randn(1, 1, N_MELS, NUM_FRAMES)
onnx_path = EXPORT_DIR / f'kws_{RUN_ID}.onnx'
torch.onnx.export(
    infer, dummy, str(onnx_path),
    input_names=['mel'], output_names=['kw_logits', 'lang_logits'],
    dynamic_axes={'mel': {0: 'batch'},
                  'kw_logits': {0: 'batch'}, 'lang_logits': {0: 'batch'}},
    opset_version=13,
)
print(f'ONNX saved: {onnx_path.name} ({onnx_path.stat().st_size / 1024:.1f} KB)')

sess = ort.InferenceSession(str(onnx_path), providers=['CPUExecutionProvider'])
kw_o, lang_o = sess.run(None, {'mel': dummy.numpy()})
print('ONNX outputs:', kw_o.shape, lang_o.shape)

In [ ]:
import shutil, subprocess, sys

try:
    import onnx2tf  # noqa: F401
except ImportError:
    print('Installing onnx2tf — this pulls TensorFlow; expect pip warnings.')
    !pip install -q onnx2tf

tf_dir = EXPORT_DIR / f'tf_{RUN_ID}'
if tf_dir.exists():
    shutil.rmtree(tf_dir)
ret = subprocess.run(
    [sys.executable, '-m', 'onnx2tf', '-i', str(onnx_path), '-o', str(tf_dir), '-osd'],
    capture_output=True, text=True,
)
print('onnx2tf return:', ret.returncode)
print(ret.stdout[-1000:])
if ret.returncode != 0:
    print('STDERR:\n', ret.stderr[-1000:])

for p in sorted(tf_dir.rglob('*.tflite')):
    print(f'  {p.relative_to(EXPORT_DIR)} — {p.stat().st_size / 1024:.1f} KB')

## 14. What's next

- **Ablate SpecAugment.** Lin et al.'s no-augmentation claim is for embedding-based models; ablating it on our DS-CNN in this small-data regime is itself a small contribution.
- **Real-data fine-tuning.** Even a few minutes of in-room recordings can close the domain gap dramatically — record yourself saying each keyword 10× and fine-tune for a couple of epochs.
- **ESP32 deployment.** Flash the int8 TFLite model + microfrontend feature extractor. Post-deadline stretch goal.
- **Report.** Three figures earn their place: the keyword × language confusion matrix on TTS test, the TTS vs. MSWC accuracy bar, and the misrouting heatmap.